# Sobre a Parte II

O **Machine Learning (ML)** amplia a análise de dados ao permitir que padrões, relações e estruturas sejam aprendidos diretamente a partir dos dados. Enquanto a Estatística fornece grande parte da base conceitual para compreender variabilidade, incerteza e inferência, o Machine Learning combina esses fundamentos com métodos computacionais de otimização e modelagem, tornando possível construir sistemas capazes de realizar previsões, classificações e outras tarefas a partir de exemplos.

Nesta segunda parte do livro, serão apresentados os principais fundamentos de Machine Learning necessários para compreender como esses modelos são construídos e treinados. Antes de estudar algoritmos específicos, discutiremos o processo de aprendizado como um **problema de otimização**, no qual os parâmetros de um modelo são ajustados de modo a minimizar uma função de perda. Conceitos como **funções de custo, descida do gradiente, taxa de aprendizado, hiperparâmetros e otimização estocástica** formarão a base para entender o treinamento dos diferentes modelos.

Um aspecto central desta parte será a distinção entre simplesmente ajustar os dados disponíveis e construir modelos capazes de **generalizar para novas observações**. Nesse contexto, serão discutidos conceitos fundamentais como conjuntos de treino, validação e teste, *underfitting*, *overfitting*, regularização e o compromisso entre **viés e variância**. Esses conceitos são essenciais para compreender por que um modelo com excelente desempenho nos dados utilizados durante o treinamento pode apresentar resultados insatisfatórios quando aplicado a dados ainda não observados.

Também será introduzida uma **visão probabilística do Machine Learning**, conectando conceitos de probabilidade, verossimilhança, inferência bayesiana e distribuições de probabilidade ao processo de aprendizado. Essa perspectiva permite compreender, por exemplo, por que determinadas funções de perda surgem naturalmente em problemas de regressão e classificação e fornece uma interpretação mais profunda para vários dos métodos utilizados posteriormente.

A partir desses fundamentos, estudaremos diferentes famílias de modelos, começando por métodos tradicionais, como **regressão linear, Ridge, regressão logística e Support Vector Machines (SVMs)**, e avançando gradualmente para **redes neurais artificiais e Deep Learning**. Serão apresentados conceitos como neurônios artificiais, funções de ativação, redes densamente conectadas, redes convolucionais, autoencoders, redes recorrentes e o algoritmo de *backpropagation*.

Sempre que possível, os conceitos matemáticos serão acompanhados de interpretações intuitivas, representações gráficas e exemplos computacionais. O objetivo não é apenas aprender a utilizar bibliotecas de Machine Learning, mas compreender **o que os modelos estão fazendo, quais hipóteses estão sendo assumidas e como o processo de aprendizado ocorre matematicamente**.

Grande parte da organização conceitual, das figuras e das discussões apresentadas nesta parte é baseada no livro adotado como referência principal para este estudo, complementado por outras referências de Machine Learning, Estatística e Deep Learning. Materiais adicionais e referências específicas serão citados ao longo dos capítulos sempre que necessários.

# Fundamentos de Machine Learning

Este capítulo reúne os conceitos fundamentais de *Machine Learning (ML)* que serão usados ao longo das próximas notas, tomando como referência principal o Capítulo 2 de *Modern Applications of Machine Learning in Quantum Sciences* {cite:p}`machinelearning2025`.

A ideia aqui não é apresentar ainda uma coleção de algoritmos, mas construir a linguagem comum por trás deles. Em particular, vamos responder a quatro perguntas:

1. **O que significa um modelo "aprender"?**
2. **Como medimos o erro de um modelo e ajustamos seus parâmetros?**
3. **Por que um bom ajuste nos dados de treino não garante boas previsões em dados novos?**
4. **Como a teoria de probabilidades ajuda a interpretar treinamento, incerteza e funções de perda?**

Assim, o capítulo é organizado em três blocos:

- **aprendizado como problema de otimização**;
- **generalização e regularização**;
- **visão probabilística de Machine Learning**.

Esses conceitos preparam o terreno para o próximo notebook, dedicado aos **modelos de Machine Learning propriamente ditos**: regressão linear e ridge, regressão logística, SVMs, redes neurais, CNNs, autoencoders e modelos autoregressivos.


## Aprendizado como um problema de otimização

Um modelo de Machine Learning pode ser visto, em sua forma mais geral, como uma função parametrizada

$$
f_\theta(\mathbf{x}),
$$

que recebe uma entrada $\mathbf{x}$ e produz uma saída interpretada como uma previsão. A forma dessa saída depende da tarefa: em **regressão**, ela pode ser um valor contínuo; em **classificação**, pode ser um rótulo ou um vetor de probabilidades associado às classes.

Por exemplo, um modelo linear pode ser escrito como

$$
f_\theta(\mathbf{x}) = \mathbf{w}^\intercal \mathbf{x} + b,
$$

com parâmetros

$$
\theta = \{\mathbf{w}, b\}.
$$

Ao escolher a forma funcional do modelo, escolhemos também uma **classe de hipóteses**: o conjunto de funções que podem ser obtidas variando os parâmetros $\theta$.

O aprendizado consiste então em procurar, dentro dessa classe, os parâmetros que tornam o modelo mais adequado aos dados. Se denotarmos o modelo treinado por

$$
\hat f \equiv f_{\theta^\ast},
$$

o treinamento pode ser formulado como

$$
\theta^\ast = \arg\min_\theta \mathcal{L}(\theta),
$$

onde $\mathcal{L}$ é uma **função de perda** (*loss function*).

> **Ideia central:** em grande parte do Machine Learning, "aprender" significa transformar o ajuste de um modelo em um problema de otimização.


### Funções de perda

A função de perda quantifica o quanto as previsões produzidas pelo modelo diferem do resultado desejado. Ela fornece o critério objetivo que o algoritmo de treinamento tentará minimizar.

Duas funções de perda aparecem repetidamente em Machine Learning:

- **Erro Quadrático Médio (MSE — Mean Squared Error)**, muito usado em regressão;
- **Entropia Cruzada (CE — Cross-Entropy)**, muito usada em classificação.

Para um conjunto de $n$ observações, o MSE é

$$
\mathcal{L}_{\mathrm{MSE}}
=
\frac{1}{n}
\sum_{i=1}^{n}
\left[
y_i - f_\theta(\mathbf{x}_i)
\right]^2.
$$

Já o **Erro Absoluto Médio (MAE — Mean Absolute Error)** é

$$
\mathcal{L}_{\mathrm{MAE}}
=
\frac{1}{n}
\sum_{i=1}^{n}
\left|
y_i - f_\theta(\mathbf{x}_i)
\right|.
$$

As duas perdas medem o erro de regressão de maneiras diferentes. O MSE cresce quadraticamente com o resíduo e, portanto, penaliza fortemente erros grandes. O MAE cresce linearmente, sendo menos dominado por grandes desvios. Para erros de módulo menor que 1, porém, $|e| > e^2$, de modo que o MAE pode atribuir uma penalização numericamente maior a pequenos resíduos.

A escolha da função de perda não é apenas uma decisão computacional. Como veremos na seção probabilística, diferentes perdas podem ser associadas a diferentes hipóteses sobre a distribuição dos dados e do ruído.


In [ ]:
import fitz  # PyMuPDF
from IPython.display import SVG, display
import re

def plot_pdf_as_svg(pdf_path, zoom=2.0, width=800, height=600):
    """
    Lê um PDF de uma página (gráfico), aplica zoom interno (matrix) no conteúdo,
    e também ajusta a tela externa (width, height) do SVG final.
    """
    doc = fitz.open(pdf_path)
    page = doc[0]

    # Cria matriz de zoom
    matrix = fitz.Matrix(zoom, zoom)

    # Gera SVG com conteúdo escalado
    svg_text = page.get_svg_image(matrix=matrix)

    # Substitui a tag <svg> para ajustar a moldura externa (pixels finais)
    svg_tag_pattern = r'<svg\b[^>]*>'
    new_svg_tag = (f'<svg xmlns="http://www.w3.org/2000/svg" '
                   f'xmlns:xlink="http://www.w3.org/1999/xlink" '
                   f'width="{width}" height="{height}">')
    svg_text = re.sub(svg_tag_pattern, new_svg_tag, svg_text, count=1)

    # Exibe
    display(SVG(svg_text))

    doc.close()


In [ ]:
plot_pdf_as_svg('../../_static/fig_2_1.pdf', zoom=2.7, width=1200, height=400)


*Figura 2.1 da Ref. {cite:p}`machinelearning2025`: exemplos de funções de perda.*

A figura reúne três ideias importantes:

**(a) Entropia cruzada binária.** Para uma observação cujo rótulo verdadeiro é $y_i=1$, a perda diminui quando a probabilidade prevista para a classe 1 se aproxima de 1. Para $y_i=0$, ocorre o comportamento oposto. Previsões muito confiantes e incorretas recebem penalizações muito grandes.

**(b) Resíduo em regressão.** As linhas tracejadas representam as diferenças entre o valor observado $y_i$ e o valor previsto $f(\mathbf{x}_i)$. As funções de perda transformam esses resíduos em um único número que será minimizado durante o treinamento.

**(c) MSE e MAE para um único ponto.** O MSE cresce como o quadrado do erro, enquanto o MAE cresce linearmente. Essa diferença explica por que o MSE é mais sensível a grandes desvios e o MAE tende a ser mais robusto à presença de *outliers*.


### Entropia cruzada

Em classificação binária, é comum interpretar a saída do modelo como uma probabilidade

$$
f_\theta(\mathbf{x}_i) \in [0,1].
$$

A **Entropia Cruzada Binária (BCE — Binary Cross-Entropy)**, também conhecida como *log loss*, é

$$
\mathcal{L}_{\mathrm{BCE}}
=
-\frac{1}{n}
\sum_{i=1}^{n}
\left[
y_i \log f_\theta(\mathbf{x}_i)
+
(1-y_i)\log\left(1-f_\theta(\mathbf{x}_i)\right)
\right].
$$

Para classificação com $K$ classes, utiliza-se frequentemente a **Entropia Cruzada Categórica (CCE)**,

$$
\mathcal{L}_{\mathrm{CCE}}
=
-\frac{1}{n}
\sum_{i=1}^{n}
\sum_{c=1}^{K}
y_{i,c}
\log p_{i,c},
$$

onde $p_{i,c}$ é a probabilidade prevista para a classe $c$ e $y_{i,c}$ é o rótulo em representação **one-hot**.

Por exemplo, para três classes, um exemplo pertencente à segunda classe pode ser escrito como

$$
\mathbf{y}_i = (0,1,0).
$$

Nesse caso, somente o termo associado à classe correta contribui diretamente para a perda.

É útil distinguir **funções de perda** de **métricas de avaliação**. Acurácia, precisão e *recall*, por exemplo, são excelentes para interpretar o desempenho final de um classificador, mas não são, em geral, funções suaves e diferenciáveis dos parâmetros. Já perdas como MSE e entropia cruzada são adequadas à otimização por gradientes.

A conexão entre **entropia cruzada**, **divergência de Kullback–Leibler** e **máxima verossimilhança** será retomada na seção de visão probabilística.


### Descida do gradiente

Depois de escolher uma função de perda, precisamos encontrar os parâmetros que a minimizam. Um dos métodos mais importantes é a **descida do gradiente** (*gradient descent*).

O gradiente

$$
\nabla_\theta \mathcal{L}
$$

aponta para a direção de maior crescimento local da função de perda. Portanto, para reduzir a perda, atualizamos os parâmetros na direção oposta:

$$
\theta_j
\leftarrow
\theta_j
-
\eta
\frac{\partial \mathcal{L}}{\partial \theta_j},
$$

onde $\eta$ é a **taxa de aprendizado** (*learning rate*).

O processo pode ser resumido em quatro etapas:

1. produzir previsões com os parâmetros atuais;
2. calcular a função de perda;
3. calcular os gradientes da perda em relação aos parâmetros;
4. atualizar os parâmetros e repetir o processo.

Cada passagem completa pelos dados de treinamento é usualmente chamada de **época** (*epoch*).

A taxa de aprendizado controla o tamanho dos passos no espaço de parâmetros. Se $\eta$ for muito pequeno, a convergência pode ser extremamente lenta. Se for muito grande, as atualizações podem ultrapassar a região de mínimo e até tornar o treinamento instável.

A taxa de aprendizado é um exemplo de **hiperparâmetro**: um valor escolhido para controlar o processo de aprendizagem. Isso a diferencia dos **parâmetros do modelo**, como pesos e *biases*, que são ajustados durante o treinamento.


### Treinamento, validação e hiperparâmetros

Como hiperparâmetros não são aprendidos da mesma forma que os parâmetros internos do modelo, precisamos de dados separados para escolhê-los.

Uma divisão conceitual importante é:

- **treino:** usado para ajustar os parâmetros do modelo;
- **validação:** usado para comparar configurações e escolher hiperparâmetros;
- **teste:** reservado para avaliar o desempenho final em dados não utilizados no processo de ajuste.

Quando a quantidade de dados é limitada, uma alternativa é a **validação cruzada $k$-fold**. O conjunto disponível é dividido em $k$ subconjuntos; em cada rodada, um subconjunto é usado para validação e os demais para treinamento. O desempenho final é então estimado pela média das rodadas.

Essa separação será retomada em mais detalhes quando discutirmos **generalização**. Por enquanto, o ponto principal é: otimizar os parâmetros e escolher hiperparâmetros são problemas diferentes, e idealmente não devem usar exatamente a mesma informação.


### Diferenciação automática

A descida do gradiente exige o cálculo de derivadas da função de perda em relação aos parâmetros. Existem diferentes maneiras de obtê-las:

- derivação analítica;
- diferenças finitas;
- **diferenciação automática (AD — Automatic Differentiation)**.

A AD explora o fato de que um programa pode ser decomposto em operações elementares cujas derivadas são conhecidas. Aplicando repetidamente a regra da cadeia,

$$
\frac{d f(g(x))}{dx}
=
f'(g(x))g'(x),
$$

é possível calcular derivadas com precisão numérica sem aproximá-las por diferenças finitas.

Nas redes neurais, a aplicação eficiente da diferenciação automática em modo reverso dá origem à **retropropagação** (*backpropagation*), que será estudada em detalhe no notebook de modelos.


### Paisagem de perda, SGD e otimizadores

A representação de $\mathcal{L}(\theta)$ no espaço de parâmetros é frequentemente chamada de **paisagem de perda** (*loss landscape*).

Se a função for convexa, a otimização é relativamente simples. Entretanto, em modelos complexos — especialmente em Deep Learning — a paisagem costuma ser altamente não convexa e pode conter muitos mínimos locais, regiões planas e pontos de sela.

Uma modificação fundamental da descida do gradiente é a **Descida do Gradiente Estocástica (SGD — Stochastic Gradient Descent)**. Em vez de calcular a perda e o gradiente usando todo o conjunto de treinamento em cada atualização, utiliza-se um **mini-batch**.

Para um mini-batch $\mathcal{B}$,

$$
\mathcal{L}_{\mathcal{B}}
=
\frac{1}{|\mathcal{B}|}
\sum_{(\mathbf{x}_i,y_i)\in\mathcal{B}}
\mathcal{L}
\left(
y_i,f_\theta(\mathbf{x}_i)
\right).
$$

Em seguida,

$$
\theta
\leftarrow
\theta
-
\eta
\nabla_\theta
\mathcal{L}_{\mathcal{B}}.
$$

A utilização de mini-batches tem dois efeitos importantes:

- reduz o custo de cada atualização para conjuntos de dados grandes;
- introduz flutuações no gradiente, o que pode ajudar a atravessar regiões planas e escapar de pontos de sela ou mínimos estreitos.

Uma forma operacional do algoritmo é:

```text
inicialize θ

para cada época:
    embaralhe os dados de treinamento

    para cada mini-batch B:
        calcule a perda em B
        calcule ∇θ L_B
        atualize θ ← θ - η ∇θ L_B
```

Vários otimizadores modernos partem dessa ideia. **Momentum** incorpora informação das atualizações anteriores; métodos adaptativos ajustam efetivamente a escala dos passos; o **Adam** combina essas duas ideias. O **L-BFGS**, por outro lado, utiliza informação aproximada de curvatura. Também existem métodos sem gradiente, como algoritmos genéticos, enxame de partículas, busca aleatória, recozimento simulado e otimização Bayesiana.


### Curvatura e Hessiana

A curvatura da função de perda em torno de um ponto $\theta^\ast$ pode ser estudada pela **matriz Hessiana**,

$$
H_{\theta^\ast}
=
\left.
\frac{\partial^2 \mathcal{L}}
{\partial\theta_i\,\partial\theta_j}
\right|_{\theta=\theta^\ast}.
$$

Se um autovalor da Hessiana é grande e positivo, a função cresce rapidamente na direção do autovetor correspondente. Autovalores próximos de zero indicam direções aproximadamente planas; autovalores negativos indicam curvatura negativa.

Em modelos de alta dimensão, é comum encontrar muitas direções quase planas. Isso ajuda a entender por que a imagem simplificada de "descer uma única bacia até o mínimo global" raramente representa adequadamente o treinamento de redes profundas.

O objetivo prático não é necessariamente encontrar o mínimo global da perda de treinamento. Um modelo pode atingir perda de treino extremamente pequena e ainda assim apresentar desempenho ruim em dados novos. Esse ponto nos leva ao conceito mais importante da próxima seção: **generalização**.


## Generalização e Regularização

Até aqui, Machine Learning poderia parecer apenas uma forma sofisticada de ajustar funções aos dados disponíveis. O aspecto que transforma esse ajuste em aprendizado útil é a **generalização**:

> **Generalização é a capacidade de produzir boas previsões para dados novos, que não participaram do treinamento.**

Idealmente, gostaríamos de medir o erro esperado do modelo sobre toda a distribuição de dados que ele encontrará no futuro. Como essa distribuição normalmente não é acessível, utilizamos um **conjunto de teste** separado como aproximação prática.

O conjunto de teste não deve ser usado:

- para ajustar os parâmetros do modelo;
- para escolher hiperparâmetros;
- para decidir qual arquitetura utilizar;
- nem para selecionar transformações de dados.

Ele serve para estimar, ao final do desenvolvimento, o desempenho em dados não vistos.

Por isso, a divisão conceitual é

$$
\mathcal{D}
=
\mathcal{D}_{\text{train}}
\cup
\mathcal{D}_{\text{val}}
\cup
\mathcal{D}_{\text{test}}.
$$

O livro sugere a proporção 8:1:1 apenas como um possível ponto de partida; a divisão adequada depende da quantidade e da estrutura dos dados.

### Vazamento de informação

Um cuidado fundamental é evitar **data leakage**: qualquer situação em que informação que deveria estar indisponível no momento da previsão acaba influenciando o treinamento.

Um exemplo clássico é normalizar todo o conjunto de dados antes da separação em treino, validação e teste. Nesse caso, estatísticas calculadas usando o conjunto de teste — como média, desvio padrão, mínimo ou máximo — passam indiretamente para o processo de treinamento.

A prática correta é ajustar transformações usando **somente o conjunto de treino** e depois aplicar os parâmetros aprendidos aos conjuntos de validação e teste.


In [ ]:
plot_pdf_as_svg('../../_static/fig_2_3.pdf', zoom=2.4, width=1050, height=380)


*Figura 2.3 da Ref. {cite:p}`machinelearning2025`: subajuste, ajuste apropriado e sobreajuste.*

A figura ilustra a relação entre a **capacidade do modelo** e a complexidade da estrutura presente nos dados.

**(a) Underfitting — subajuste.** O modelo é simples demais para representar a estrutura dos dados. Ele apresenta erro alto mesmo no conjunto de treinamento. Dizemos que o modelo possui capacidade insuficiente.

**(b) Ajuste apropriado.** A fronteira possui flexibilidade suficiente para capturar a estrutura relevante sem acompanhar cada detalhe particular das observações de treino. Esse é o regime desejado.

**(c) Overfitting — sobreajuste.** O modelo possui capacidade excessiva e começa a acomodar peculiaridades do conjunto de treinamento, inclusive ruído. O erro de treino pode continuar diminuindo, enquanto o desempenho em dados novos piora.

Portanto, **menor erro de treinamento não implica necessariamente melhor modelo**. O objetivo é encontrar uma função que capture regularidades que se mantenham fora da amostra de treino.


### Capacidade e regularização

A **capacidade** de um modelo pode ser entendida, de forma aproximada, como sua habilidade de representar uma variedade de funções.

- capacidade muito baixa $\rightarrow$ **underfitting**;
- capacidade excessiva $\rightarrow$ maior risco de **overfitting**;
- capacidade adequada $\rightarrow$ melhor compromisso entre ajuste e generalização.

Chamamos de **regularização** qualquer modificação destinada a melhorar a generalização, mesmo que isso possa aumentar o erro de treinamento.

Uma maneira clássica de regularizar é penalizar parâmetros muito grandes. Se a perda original é $\mathcal{L}_{\text{train}}$, podemos escrever

$$
\mathcal{L}_{\text{reg}}
=
\mathcal{L}_{\text{train}}
+
\lambda \,\Omega(\theta),
$$

onde $\lambda$ controla a intensidade da regularização.

Duas escolhas importantes são:

$$
\Omega_{L_2}(\theta)
=
\|\theta\|_2^2
=
\sum_j \theta_j^2,
$$

e

$$
\Omega_{L_1}(\theta)
=
\|\theta\|_1
=
\sum_j |\theta_j|.
$$

A penalização $L_2$ desencoraja coeficientes de grande magnitude. A penalização $L_1$ pode favorecer soluções esparsas, em que vários coeficientes se tornam exatamente ou aproximadamente nulos.

Essas ideias aparecerão novamente na **regressão ridge** e no **LASSO**.

A motivação pode ser relacionada à **navalha de Occam**: entre explicações que descrevem adequadamente os dados observados, há boas razões para favorecer soluções mais simples. Ao mesmo tempo, o teorema *no free lunch* lembra que não existe um algoritmo universalmente superior em todos os problemas possíveis. A regularização introduz hipóteses sobre quais soluções esperamos que funcionem melhor para a tarefa específica.


### Trade-off entre viés e variância

A relação entre complexidade e generalização pode ser formalizada pelo **trade-off viés–variância**.

Considere um problema de regressão em que

$$
y = f(x) + \varepsilon,
$$

com $\varepsilon$ representando o ruído dos dados. Para um ponto $x_0$, usando MSE, o erro esperado de previsão pode ser decomposto como

$$
\operatorname{Err}(x_0)
=
\operatorname{Bias}^2[\hat f(x_0)]
+
\operatorname{Var}[\hat f(x_0)]
+
\mathbb{E}[\varepsilon^2].
$$

O **viés ao quadrado** é

$$
\operatorname{Bias}^2[\hat f(x_0)]
=
\left(
\mathbb{E}[\hat f(x_0)] - f(x_0)
\right)^2,
$$

e mede o erro sistemático entre a previsão média do modelo e a função verdadeira.

A **variância** é

$$
\operatorname{Var}[\hat f(x_0)]
=
\mathbb{E}
\left[
\left(
\hat f(x_0)
-
\mathbb{E}[\hat f(x_0)]
\right)^2
\right],
$$

e mede o quanto o modelo mudaria se fosse treinado com diferentes amostras extraídas do mesmo processo gerador de dados.

O último termo,

$$
\mathbb{E}[\varepsilon^2],
$$

representa a variância do ruído inerente aos dados. Do ponto de vista do modelo, essa componente é **irredutível**.

Em termos intuitivos:

- modelos simples tendem a apresentar **alto viés e baixa variância**;
- modelos muito flexíveis tendem a apresentar **baixo viés e alta variância**.

O objetivo é encontrar um compromisso que minimize o erro esperado em dados não vistos.


In [ ]:
plot_pdf_as_svg('../../_static/fig_2_4.pdf', zoom=2.4, width=1050, height=480)


*Figura 2.4 da Ref. {cite:p}`machinelearning2025`: trade-off entre viés e variância.*

A curva de erro de treinamento tende a cair à medida que a complexidade do modelo aumenta: um modelo mais flexível consegue se adaptar cada vez melhor aos dados usados no ajuste.

O erro de teste, entretanto, apresenta um comportamento diferente. Inicialmente ele diminui, porque o modelo deixa de ser simples demais. Depois atinge uma região de mínimo. A partir daí, aumentar ainda mais a capacidade pode fazer o modelo adaptar-se excessivamente às particularidades do treinamento, elevando novamente o erro de teste.

Assim, podemos associar três regimes:

- **baixa complexidade:** alto viés, baixa variância e underfitting;
- **complexidade intermediária:** menor erro de generalização;
- **alta complexidade:** baixo viés, alta variância e maior risco de overfitting.

A figura deve ser interpretada como uma visão conceitual clássica. Em redes neurais modernas superparametrizadas, a relação entre capacidade e generalização pode ser mais complexa do que esse esquema simples sugere.


## Visão probabilística em Machine Learning

A formulação por otimização nos diz **como** ajustar um modelo. A visão probabilística ajuda a entender **o que está sendo estimado** e como incorporar **incerteza**.

O livro destaca três fontes gerais de incerteza:

1. **estocasticidade inerente ao processo gerador dos dados**;
2. **observabilidade incompleta**, quando não temos acesso a todas as variáveis relevantes;
3. **modelagem incompleta**, pois todo modelo retém algumas informações e descarta outras.

A linguagem de probabilidades fornece uma forma natural de representar essas limitações.

### Variáveis aleatórias e distribuições

Uma variável aleatória $X$ pode assumir diferentes valores $x$ segundo uma distribuição de probabilidade.

Para variáveis discretas usamos uma função de massa de probabilidade; para variáveis contínuas, uma densidade de probabilidade. Em notação compacta, escrevemos simplesmente

$$
p(X=x) \equiv p(x).
$$

Para duas variáveis,

$$
p(x,y)
$$

representa sua distribuição conjunta.

Se $X$ e $Y$ forem independentes,

$$
p(x,y)
=
p(x)p(y).
$$

Essa hipótese aparece frequentemente em Machine Learning, especialmente quando assumimos observações **independentes e identicamente distribuídas (i.i.d.)**.


### Probabilidade condicional e regra da cadeia

A probabilidade de $Y=y$ sabendo que $X=x$ é

$$
p(y|x)
=
\frac{p(x,y)}{p(x)}.
$$

Para várias variáveis, uma distribuição conjunta pode ser fatorada usando a **regra da cadeia da probabilidade**:

$$
p\left(x^{(1)},\ldots,x^{(n)}\right)
=
p\left(x^{(1)}\right)
\prod_{i=2}^{n}
p\left(
x^{(i)}
\mid
x^{(1)},\ldots,x^{(i-1)}
\right).
$$

Essa decomposição é especialmente importante em modelos autoregressivos, nos quais uma distribuição complexa é construída como produto de distribuições condicionais.


### Teorema de Bayes

O teorema de Bayes permite inverter uma probabilidade condicional:

$$
p(x|y)
=
\frac{p(y|x)\,p(x)}{p(y)}.
$$

Na interpretação Bayesiana:

- $p(x)$ é o **prior**: conhecimento ou crença antes de observar a evidência;
- $p(y|x)$ é a **verossimilhança** (*likelihood*);
- $p(y)$ é a **evidência** ou constante de normalização;
- $p(x|y)$ é o **posterior**: conhecimento atualizado depois de observar $y$.

Esquematicamente,

$$
\text{posterior}
\propto
\text{likelihood}
\times
\text{prior}.
$$

Essa ideia será importante mais adiante quando compararmos **máxima verossimilhança (MLE)** e **máximo a posteriori (MAP)**. Na regressão ridge, por exemplo, a regularização $L_2$ poderá ser interpretada como consequência da introdução de um prior Gaussiano sobre os parâmetros.


### Aprendizado supervisionado e não supervisionado em linguagem probabilística

A visão probabilística permite reinterpretar os dois principais cenários de aprendizado.

No **aprendizado não supervisionado**, observamos exemplos

$$
\mathbf{x}_1,\mathbf{x}_2,\ldots,\mathbf{x}_n
$$

e buscamos aprender a distribuição

$$
p(\mathbf{x})
$$

ou alguma de suas propriedades.

No **aprendizado supervisionado**, observamos pares

$$
(\mathbf{x}_i,y_i)
$$

e buscamos aprender como a variável de saída depende da entrada. Em classificação, isso pode ser entendido como a estimação de

$$
p(y|\mathbf{x}).
$$

Se conhecêssemos exatamente essa distribuição condicional, o classificador de Bayes escolheria

$$
y_{\text{Bayes}}
=
\arg\max_y
p(y|\mathbf{x}).
$$

Esse classificador é ótimo no sentido de minimizar a probabilidade de erro de classificação sob a distribuição verdadeira. Ainda assim, seu erro pode ser diferente de zero.

Esse limite fundamental é chamado de **erro de Bayes**. Ele representa a parcela de ambiguidade intrínseca ao problema que não pode ser eliminada simplesmente escolhendo um modelo mais sofisticado.


### Verossimilhança

Considere agora um modelo parametrizado por $\theta$ e um conjunto observado de dados $\mathcal{D}$.

A **verossimilhança**

$$
p(\mathcal{D}|\theta)
$$

mede quão compatíveis os dados observados são com diferentes valores dos parâmetros do modelo.

É importante não confundir **probabilidade** e **verossimilhança**:

- em $p(\mathcal{D}|\theta)$, se $\theta$ está fixo, a expressão pode ser interpretada como distribuição sobre possíveis dados;
- na inferência de parâmetros, mantemos $\mathcal{D}$ fixo e estudamos a mesma expressão como função de $\theta$.

O princípio de **máxima verossimilhança (MLE)** escolhe

$$
\theta_{\text{MLE}}
=
\arg\max_\theta
p(\mathcal{D}|\theta).
$$

Como o logaritmo é monotônico, isso é equivalente a

$$
\theta_{\text{MLE}}
=
\arg\max_\theta
\log p(\mathcal{D}|\theta).
$$

Essa conexão é central porque muitas funções de perda usadas em ML correspondem, essencialmente, ao **negativo do logaritmo de uma verossimilhança**.


### Divergência de Kullback–Leibler e entropia cruzada

Para comparar duas distribuições $p$ e $q$, podemos utilizar a **divergência de Kullback–Leibler (KL)**:

$$
D_{\mathrm{KL}}(p\|q)
=
\sum_x
p(x)
\log
\frac{p(x)}{q(x)}.
$$

No caso contínuo, a soma é substituída por uma integral.

A KL é não negativa e vale zero quando as distribuições coincidem, mas não é uma distância propriamente dita, pois em geral

$$
D_{\mathrm{KL}}(p\|q)
\neq
D_{\mathrm{KL}}(q\|p).
$$

Separando o logaritmo,

$$
D_{\mathrm{KL}}(p\|q)
=
\sum_x p(x)\log p(x)
-
\sum_x p(x)\log q(x).
$$

Definindo a entropia de Shannon

$$
S(p)
=
-
\sum_x
p(x)\log p(x),
$$

e a entropia cruzada

$$
H(p,q)
=
-
\sum_x
p(x)\log q(x),
$$

temos

$$
D_{\mathrm{KL}}(p\|q)
=
-H(p,p)+H(p,q)
=
-S(p)+H(p,q).
$$

Como $S(p)$ não depende do modelo $q$, minimizar

$$
D_{\mathrm{KL}}(p\|q)
$$

em relação a $q$ é equivalente a minimizar a **entropia cruzada**.

Isso explica por que a entropia cruzada aparece tão naturalmente na classificação: o treinamento procura aproximar a distribuição-alvo pelas probabilidades produzidas pelo modelo.


### Fechando o ciclo: probabilidade, perda e otimização

Agora podemos conectar as três perspectivas discutidas neste capítulo.

Em Machine Learning supervisionado, dispomos de dados e escolhemos uma família parametrizada de modelos. O treinamento ajusta os parâmetros por otimização, mas as funções de perda frequentemente possuem uma interpretação probabilística:

- **entropia cruzada** aparece naturalmente ao comparar distribuições de classe e pode ser entendida pela KL ou pela máxima verossimilhança;
- **MSE** emerge quando modelamos o alvo como uma variável Gaussiana em torno da previsão do modelo;
- **regularização** pode ser interpretada, em certos casos, como a introdução de informação *a priori* sobre os parâmetros.

Portanto, as ideias não são independentes:

$$
\boxed{
\text{probabilidade}
\longrightarrow
\text{função de perda}
\longrightarrow
\text{otimização}
\longrightarrow
\text{modelo treinado}
}
$$

mas um modelo treinado só é realmente útil se a solução encontrada **generalizar** para dados novos.

Com isso, temos a base conceitual necessária para estudar modelos específicos. No próximo notebook, começaremos pela **regressão linear e ridge** e seguiremos para classificação, SVMs e redes neurais.
